In [53]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [54]:
df=pd.read_csv('DateFruit_Dataset.csv')
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [55]:
X=df.drop('Class',axis=1)
y=df['Class']

In [56]:
le=LabelEncoder()
y=le.fit_transform(y)

In [57]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [58]:
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [140]:
from sklearn.decomposition import PCA
pca=PCA(n_components=28)
X_train_pca=pca.fit_transform(X_train_scaled)
X_test_pca=pca.transform(X_test_scaled)

In [141]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset

In [142]:
X_train_tensor=torch.tensor(X_train_pca,dtype=torch.float32)
X_test_tensor=torch.tensor(X_test_pca, dtype=torch.float32)
y_train_tensor=torch.tensor(y_train,dtype=torch.long)
y_test_tensor=torch.tensor(y_test,dtype=torch.long)

In [143]:
train_dataset=TensorDataset(X_train_tensor,y_train_tensor)
test_dataset=TensorDataset(X_test_tensor,y_test_tensor)

In [144]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32)

In [145]:
class ANN(nn.Module):
    def __init__(self):
      super(ANN,self).__init__()

      self.model=nn.Sequential(
            nn.Linear(X_train_pca.shape[1],64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7)  
           )
    def forward(self,x):
        return self.model(x)

In [146]:
model=ANN()
#LOSS AND OPTIM
criteria=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [147]:
#train the model

epochs=100
for epoch in range(epochs):
    model.train()

    running_loss=0.0
    for Xb,yb in train_loader:
        optimizer.zero_grad()
        outputs=model(Xb)
        loss=criteria(outputs,yb)
        loss.backward()
        optimizer.step()
        running_loss +=loss.item()
    train_loss=running_loss/len(train_loader)    
    print(f'epoch={epoch+1}/{epochs},loss={train_loss}')

epoch=1/100,loss=1.7344394766766091
epoch=2/100,loss=1.2646987515947092
epoch=3/100,loss=0.8766885140667791
epoch=4/100,loss=0.6361421916795813
epoch=5/100,loss=0.4895001131555308
epoch=6/100,loss=0.3939799536829409
epoch=7/100,loss=0.33309932538996573
epoch=8/100,loss=0.2921077025973279
epoch=9/100,loss=0.26222437166649365
epoch=10/100,loss=0.24697944262753363
epoch=11/100,loss=0.2180284760568453
epoch=12/100,loss=0.2017439402963804
epoch=13/100,loss=0.19205602193656174
epoch=14/100,loss=0.18077722936868668
epoch=15/100,loss=0.16589275763734526
epoch=16/100,loss=0.1616983484962712
epoch=17/100,loss=0.15320139929004337
epoch=18/100,loss=0.1418112127677254
epoch=19/100,loss=0.13790894477911617
epoch=20/100,loss=0.13653853859590448
epoch=21/100,loss=0.13516658609328064
epoch=22/100,loss=0.12416582554578781
epoch=23/100,loss=0.11669820247460967
epoch=24/100,loss=0.12198065188915833
epoch=25/100,loss=0.10829326186491095
epoch=26/100,loss=0.10882476733430572
epoch=27/100,loss=0.101354849403

In [148]:
#Evaluate 
model.eval()
total=0
correct=0

with torch.no_grad():
    for Xb,yb in test_loader:
        outputs=model(Xb)
        _,predicted=torch.max(outputs,1)

        correct+=(predicted==yb).sum().item()
        total+=yb.size(0)  #actual samples in batch
print('accuracy',correct/total*100)  

accuracy 96.11111111111111
